# GOES-R series (GOES-16, -17, -18, -19) / ABI (Advanced Baseline Imager)

Example notebook from the [Satellite Data Catalog](https://alessandrosebastianelli.github.io/satcat/satellite.html?folder=noaa/goes-r-abi). Each section below runs the Python example of one processing level.

Run the cells top to bottom. Some sources need a free account; the cell that needs credentials asks for them when it runs.

## L1b

Level 1b: calibrated radiances (Rad) on the ABI fixed grid, for full disc (RadF), CONUS (RadC) and mesoscale (RadM) sectors.

Reads one full-disc ABI radiance file from the public NOAA bucket on AWS (no credentials), converts band 13 to brightness temperature and plots it.

In [ ]:
%pip install -q s3fs xarray h5netcdf matplotlib numpy

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)                # public NOAA bucket, no credentials needed
BUCKET = "noaa-goes19"                               # noaa-goes16, noaa-goes18, noaa-goes19 (GOES-17 until 2023)
PRODUCT = "ABI-L1b-RadF"                             # F = full disk, C = CONUS, M = mesoscale
day = dt.datetime(2025, 6, 1, 12)
prefix = "%s/%s/%s/%s/%s/" % (BUCKET, PRODUCT, day.strftime("%Y"), day.strftime("%j"), day.strftime("%H"))
files = [f for f in fs.ls(prefix) if "C13_G" in f]
print(len(files), "files in", prefix)
if not files:
    raise SystemExit("No files: check the bucket (satellite) and the date")

with fs.open(files[0]) as fobj:
    ds = xr.open_dataset(fobj, engine="h5netcdf").load()
print(ds)

# Radiance -> brightness temperature (IR bands) using the constants stored in the file
rad = ds["Rad"]
bt = (ds["planck_fk2"] / np.log(ds["planck_fk1"] / rad + 1) - ds["planck_bc1"]) / ds["planck_bc2"]
bt[::4, ::4].plot(cmap="gray_r", robust=True)
plt.title("GOES ABI band 13 brightness temperature (K)")
plt.show()

## L2

Level 2: Cloud and Moisture Imagery (CMIP, MCMIP multi-band) and derived products: cloud mask, cloud top height, SST, LST, fire (FDC), aerosol optical depth, rainfall rate and others.

Reads one full-disc cloud and moisture imagery file (band 13) from the public NOAA bucket on AWS (no credentials) and plots it.

In [ ]:
%pip install -q s3fs xarray h5netcdf matplotlib numpy

In [ ]:
import datetime as dt

import matplotlib.pyplot as plt
import numpy as np
import s3fs
import xarray as xr

fs = s3fs.S3FileSystem(anon=True)                # public NOAA bucket, no credentials needed
BUCKET = "noaa-goes19"                               # noaa-goes16, noaa-goes18, noaa-goes19 (GOES-17 until 2023)
PRODUCT = "ABI-L2-CMIPF"                             # F = full disk, C = CONUS, M = mesoscale
day = dt.datetime(2025, 6, 1, 12)
prefix = "%s/%s/%s/%s/%s/" % (BUCKET, PRODUCT, day.strftime("%Y"), day.strftime("%j"), day.strftime("%H"))
files = [f for f in fs.ls(prefix) if "C13_G" in f]
print(len(files), "files in", prefix)
if not files:
    raise SystemExit("No files: check the bucket (satellite) and the date")

with fs.open(files[0]) as fobj:
    ds = xr.open_dataset(fobj, engine="h5netcdf").load()
print(ds)

cmi = ds["CMI"]
cmi[::4, ::4].plot(cmap="gray_r", robust=True)    # band 13 brightness temperature (K) on the ABI fixed grid
plt.title("GOES ABI CMI band 13 (10.3 um)")
plt.show()